In [ ]:
import numpy as np, pandas as pd, glob
import matplotlib.pyplot as plt

fid, tile, window, i = "51", "tile_0", "evt", 0   # i = token

d = glob.glob(f"embeddings/joint_forest/fid_{fid}/{window}/*")[0]
print(d)
locs = pd.read_csv(f"{d}/{tile}_locs.csv")
samp = pd.read_csv(f"{d}/{tile}_samples.csv").to_numpy()

cell = int(locs["cell"].iloc[i]) - 1          # 1-based -> 0-based
print(cell)
row, col = cell // 15, cell % 15
forest_vec = samp[i]

print(f"FOREST · token {i} · celda {cell+1}  (fila {row}, col {col})")
print(forest_vec[:10], "...")                 # first 10 values

grid = np.zeros((15, 15)); grid[row, col] = 1
plt.figure(figsize=(4, 4))
plt.imshow(grid, cmap="Greens")
plt.title(f"FOREST · celda {cell+1} (fila {row}, col {col})")
plt.xticks(range(15)); plt.yticks(range(15)); plt.grid(False)
plt.show()

In [ ]:
pair = d.split("/")[-1]                        # misma fecha
emb = np.load(f"embeddings/joint/fid_{fid}/{window}/{pair}/{tile}.npy").reshape(-1, 768)  # (225, 768)
print(emb[0])
print(f"embeddings/joint/fid_{fid}/{window}/{pair}/{tile}.npy")

joint_vec = emb[cell]                          # misma celda que forest
print(f"JOINT · celda {cell+1}  (fila {row}, col {col})")
print(joint_vec[:10], "...")                   # primeros 10 valores

print("\nsame", np.allclose(forest_vec, joint_vec, atol=1e-3, equal_nan=True))

grid = np.zeros((15, 15)); grid[row, col] = 1
plt.figure(figsize=(4, 4))
plt.imshow(grid, cmap="Blues")
plt.title(f"JOINT · celda {cell+1} (fila {row}, col {col})")
plt.xticks(range(15)); plt.yticks(range(15)); plt.grid(False)
plt.show()

In [ ]:
import rasterio

# el _samples.tif es el mismo tile (mismo extent que el .npy de joint) y trae el geotransform
with rasterio.open(f"{d}/{tile}_samples.tif") as s:
    T = s.transform          # relación coordenada <-> píxel
    print("bounds:", s.bounds, "| tamaño:", s.width, "x", s.height)

x, y = locs["x"].iloc[i], locs["y"].iloc[i]
row_xy, col_xy = rasterio.transform.rowcol(T, x, y)     # coordenada -> (fila, columna)